# Phase 8: SQL Business Analytics Layer

**Project:** Customer Churn Prediction & Retention Analytics  
**Database:** SQLite database at `data/processed/customer_churn.db`  
**Table:** `customer_churn` ($N = 7,043$ customer records)  
**SQL Source File:** `sql/churn_analysis.sql`  

### Objectives:
1. Ingest the prioritized retention recommendations dataset into a local relational SQLite database.
2. Execute SQL business queries covering portfolio KPIs, segment breakdowns, high-risk driver prevalence, and top retention priorities.
3. Export key SQL query results to formatted CSV reports.
4. Formally validate that SQL aggregation results match our previous Python EDA baselines with zero discrepancies.

## 1. Connect to SQLite Database

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

db_path = Path("../data/processed/customer_churn.db")
conn = sqlite3.connect(db_path)
print(f"Connected to SQLite database: {db_path.resolve()}")

def run_query(sql_query, title=None):
    """Helper to execute SQL query, display formatted table, and return DataFrame."""
    if title:
        print(f"=== {title} ===")
    result = pd.read_sql_query(sql_query, conn)
    display(result)
    return result

## 2. Query A: Overall Customer Portfolio KPIs

In [ ]:
query_a = """
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN "Churn Value" = 1 THEN 1 ELSE 0 END) AS churned_customers,
    ROUND(SUM(CASE WHEN "Churn Value" = 1 THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS churn_rate_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charges,
    ROUND(AVG("Tenure Months"), 2) AS avg_tenure_months
FROM customer_churn;
"""
res_a = run_query(query_a, "Query A: Overall KPIs")

## 3. Query B: Churn by Contract Agreement Type

In [ ]:
query_b = """
SELECT
    "Contract" AS contract_type,
    COUNT(*) AS customer_count,
    SUM(CASE WHEN "Churn Value" = 1 THEN 1 ELSE 0 END) AS churned_count,
    ROUND(SUM(CASE WHEN "Churn Value" = 1 THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS churn_rate_pct
FROM customer_churn
GROUP BY "Contract"
ORDER BY churn_rate_pct DESC;
"""
res_b = run_query(query_b, "Query B: Churn by Contract")

## 4. Query C: Churn by Internet Service Tier

In [ ]:
query_c = """
SELECT
    "Internet Service" AS internet_service_tier,
    COUNT(*) AS customer_count,
    SUM(CASE WHEN "Churn Value" = 1 THEN 1 ELSE 0 END) AS churned_count,
    ROUND(SUM(CASE WHEN "Churn Value" = 1 THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS churn_rate_pct
FROM customer_churn
GROUP BY "Internet Service"
ORDER BY churn_rate_pct DESC;
"""
res_c = run_query(query_c, "Query C: Churn by Internet Service")

## 5. Query D: Churn by Billing Payment Method

In [ ]:
query_d = """
SELECT
    "Payment Method" AS payment_method,
    COUNT(*) AS customer_count,
    SUM(CASE WHEN "Churn Value" = 1 THEN 1 ELSE 0 END) AS churned_count,
    ROUND(SUM(CASE WHEN "Churn Value" = 1 THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS churn_rate_pct
FROM customer_churn
GROUP BY "Payment Method"
ORDER BY churn_rate_pct DESC;
"""
res_d = run_query(query_d, "Query D: Churn by Payment Method")

## 6. Query E: Churn by Tenure Cohort

In [ ]:
query_e = """
SELECT
    CASE
        WHEN "Tenure Months" <= 12 THEN '0-12 months'
        WHEN "Tenure Months" <= 24 THEN '13-24 months'
        WHEN "Tenure Months" <= 36 THEN '25-36 months'
        WHEN "Tenure Months" <= 48 THEN '37-48 months'
        WHEN "Tenure Months" <= 60 THEN '49-60 months'
        ELSE '61+ months'
    END AS tenure_group,
    COUNT(*) AS customer_count,
    SUM(CASE WHEN "Churn Value" = 1 THEN 1 ELSE 0 END) AS churned_count,
    ROUND(SUM(CASE WHEN "Churn Value" = 1 THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS churn_rate_pct
FROM customer_churn
GROUP BY tenure_group
ORDER BY MIN("Tenure Months") ASC;
"""
res_e = run_query(query_e, "Query E: Churn by Tenure Cohort")

## 7. Query F: Analytical Risk Segment Validation

In [ ]:
query_f = """
SELECT
    "Risk Level" AS risk_level,
    COUNT(*) AS customer_count,
    SUM(CASE WHEN "Churn Value" = 1 THEN 1 ELSE 0 END) AS actual_churned_count,
    ROUND(SUM(CASE WHEN "Churn Value" = 1 THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS actual_churn_rate_pct,
    ROUND(AVG("Churn Probability") * 100, 2) AS avg_churn_probability_pct,
    ROUND(AVG("Monthly Charges"), 2) AS avg_monthly_charges,
    ROUND(AVG("Tenure Months"), 2) AS avg_tenure_months
FROM customer_churn
GROUP BY "Risk Level"
ORDER BY CASE 
    WHEN "Risk Level" = 'High Risk' THEN 1 
    WHEN "Risk Level" = 'Medium Risk' THEN 2 
    ELSE 3 
END ASC;
"""
res_f = run_query(query_f, "Query F: Risk Segment Analysis")

## 8. Query G: High-Risk Cohort Profile Characteristics

In [ ]:
query_g = """
SELECT
    COUNT(*) AS total_high_risk_customers,
    SUM(CASE WHEN "Contract" = 'Month-to-month' THEN 1 ELSE 0 END) AS month_to_month_count,
    ROUND(SUM(CASE WHEN "Contract" = 'Month-to-month' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS month_to_month_pct,
    SUM(CASE WHEN "Tech Support" = 'No' THEN 1 ELSE 0 END) AS no_tech_support_count,
    ROUND(SUM(CASE WHEN "Tech Support" = 'No' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS no_tech_support_pct,
    SUM(CASE WHEN "Online Security" = 'No' THEN 1 ELSE 0 END) AS no_online_security_count,
    ROUND(SUM(CASE WHEN "Online Security" = 'No' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS no_online_security_pct,
    SUM(CASE WHEN "Payment Method" = 'Electronic check' THEN 1 ELSE 0 END) AS electronic_check_count,
    ROUND(SUM(CASE WHEN "Payment Method" = 'Electronic check' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS electronic_check_pct,
    SUM(CASE WHEN "Internet Service" = 'Fiber optic' THEN 1 ELSE 0 END) AS fiber_optic_count,
    ROUND(SUM(CASE WHEN "Internet Service" = 'Fiber optic' THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS fiber_optic_pct,
    SUM(CASE WHEN "Tenure Months" <= 12 THEN 1 ELSE 0 END) AS short_tenure_count,
    ROUND(SUM(CASE WHEN "Tenure Months" <= 12 THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS short_tenure_pct
FROM customer_churn
WHERE "Risk Level" = 'High Risk';
"""
res_g = run_query(query_g, "Query G: High-Risk Profile")

## 9. Query H: Top 20 Retention Priority Accounts

In [ ]:
query_h = """
SELECT
    "Priority Rank" AS priority_rank,
    "CustomerID" AS customer_id,
    "Churn Probability" AS churn_probability,
    "Risk Level" AS risk_level,
    "Contract" AS contract,
    "Tenure Months" AS tenure_months,
    "Monthly Charges" AS monthly_charges,
    "Risk Drivers" AS risk_drivers,
    "Retention Action" AS retention_action
FROM customer_churn
ORDER BY "Priority Rank" ASC
LIMIT 20;
"""
res_h = run_query(query_h, "Query H: Top 20 Priorities")

## 10. Query I: High-Risk Customers by Contract Type

In [ ]:
query_i = """
SELECT
    "Contract" AS contract_type,
    COUNT(*) AS high_risk_customer_count,
    ROUND(AVG("Churn Probability") * 100, 2) AS avg_churn_probability_pct,
    ROUND(SUM(CASE WHEN "Churn Value" = 1 THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS actual_churn_rate_pct
FROM customer_churn
WHERE "Risk Level" = 'High Risk'
GROUP BY "Contract"
ORDER BY high_risk_customer_count DESC;
"""
res_i = run_query(query_i, "Query I: High-Risk by Contract")

## 11. Query J: High-Risk Customers by Payment Method

In [ ]:
query_j = """
SELECT
    "Payment Method" AS payment_method,
    COUNT(*) AS high_risk_customer_count,
    ROUND(AVG("Churn Probability") * 100, 2) AS avg_churn_probability_pct,
    ROUND(SUM(CASE WHEN "Churn Value" = 1 THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100, 2) AS actual_churn_rate_pct
FROM customer_churn
WHERE "Risk Level" = 'High Risk'
GROUP BY "Payment Method"
ORDER BY high_risk_customer_count DESC;
"""
res_j = run_query(query_j, "Query J: High-Risk by Payment Method")

## 12. Verification & Validation Against Python EDA Baseline

We verify that SQL database aggregations align perfectly with our EDA metrics:

In [ ]:
# 1. Portfolio KPIs Check
assert int(res_a.loc[0, 'total_customers']) == 7043
assert int(res_a.loc[0, 'churned_customers']) == 1869
assert float(res_a.loc[0, 'churn_rate_pct']) == 26.54

# 2. Contract Rates Check
contract_map = res_b.set_index('contract_type')['churn_rate_pct'].to_dict()
assert contract_map['Month-to-month'] == 42.71
assert contract_map['One year'] == 11.27
assert contract_map['Two year'] == 2.83

# 3. Risk Segment Actual Rates Check
risk_map = res_f.set_index('risk_level')['actual_churn_rate_pct'].to_dict()
assert risk_map['Low Risk'] == 4.91
assert risk_map['Medium Risk'] == 22.76
assert risk_map['High Risk'] == 58.10

print("All assertions passed! SQL business analytics results match Python EDA exactly.")